# ARC-AGI-3 — Chronos v15: three-pass self-training agent

**Setup (one-time):** upload a private Kaggle dataset named **`v15-plm`** containing EXACTLY:
- `plm/` (the whole package — includes `ttt.py`, the test-time trainer)
- `my_agent.py` (the entry point: scout → TTT → plan, all inside)
- `plm_weights.pt` (the offline-trained prior — REQUIRED for real performance)

Attach that dataset + the competition data. Accelerator: **GPU** (TTT trains in-episode). Internet: **OFF**.

**INTEGRITY CHECKLIST — the dataset must NOT contain:** solution caches (`*_bfs_cache_*.json`), engine sources, `v15_scratch/` scratchpads, or any `v13` files. The agent's pass-1 BFS finds no engine on the hidden eval and degrades to the scout — by design. Everything offline arrives only as weights.

**Runtime behavior on the hidden eval (per game, blind):**
1. pass 1 — BFS probe fails instantly (no engine) → bandit scout explores `V15_SCOUT_ACTIONS` real actions, recording every transition
2. pass 2 — TTT: finetunes belief+simulator+value on this game's own transitions (`V15_TTT_SECONDS`, costs zero actions)
3. pass 3 — value-guided latent beam search with deep-think escalation; re-scouts + retrains when stuck

In [ ]:
# Competition environment wheels (torch is preinstalled on Kaggle)
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
# Stage the v15 code + weights from the attached dataset, then sanity-check.
import os
!cp -r /kaggle/input/v15-plm/plm /kaggle/working/plm
!cp /kaggle/input/v15-plm/my_agent.py /kaggle/working/my_agent.py
!cp /kaggle/input/v15-plm/plm_weights.pt /kaggle/working/plm_weights.pt 2>/dev/null || echo 'WARNING: no weights in dataset - agent will TTT from scratch (much weaker)'

# paste-mangling / truncation guard (catches bad copies in 2 seconds)
import ast
for f in ['/kaggle/working/my_agent.py'] + \
         [f'/kaggle/working/plm/{m}' for m in os.listdir('/kaggle/working/plm') if m.endswith('.py')]:
    ast.parse(open(f).read()); print('syntax OK:', f)

# weights must exist, have all 3 keys, and be finite (the NaN lesson)
if os.path.exists('/kaggle/working/plm_weights.pt'):
    import torch
    s = torch.load('/kaggle/working/plm_weights.pt', map_location='cpu', weights_only=True)
    assert set(s) >= {'tokenizer', 'belief', 'world_model'}, f'missing keys: {set(s)}'
    for part, sd in s.items():
        for k, v in sd.items():
            assert torch.isfinite(v).all(), f'NaN/Inf in {part}/{k}'
    print('weights OK:', list(s))

# full module smoke test (interactive runs only - skip during scoring rerun)
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !cd /kaggle/working && python -m plm.smoke

In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # wait for the evaluation gateway, then stage the official harness
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    # agent file + the plm package + weights, all side by side so
    # my_agent.py's sys.path.insert(dirname(__file__)) finds the package
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    !cp -r /kaggle/working/plm /kaggle/working/ARC-AGI-3-Agents/agents/templates/plm
    !cp /kaggle/working/plm_weights.pt /kaggle/working/ARC-AGI-3-Agents/agents/templates/plm_weights.pt 2>/dev/null || true
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py','w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
""")
    with open('/kaggle/working/ARC-AGI-3-Agents/.env','w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    # Three-pass budgets for the T4 (see KAGGLE.md for the reasoning).
    # PYTHONUNBUFFERED -> real-time Logs tab; tee -> downloadable artifact
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg PYTHONUNBUFFERED=1 \
        V15_SCOUT_ACTIONS=80 V15_TTT_SECONDS=180 V15_THINK_BUDGET=120 \
        V15_RESCOUT_ACTIONS=40 V15_STUCK_WINDOW=60 \
        python main.py --agent myagent 2>&1 | tee /kaggle/working/v15_run.log

The cell above only runs during the competition scoring rerun, not in interactive tests.

**What to look for in the Logs tab (Save & Run All test):** `V15 agent ready: plm=on torch=yes phase=scout`, then per game: `pass1-BFS: no engine/v13 module — scout path` (expected!), `scout(N left):...` actions, `V15 TTT #1: training on ...` followed by `V15 TTT done: {...}`, then `plm:bfs-soft(p=...)` / `plm:think(...)` reasonings. `plm=OFF` means the dataset/weights didn't stage — fix before submitting.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet',index=False)

This is a dummy submission fallback, important to keep.